# MC-Dropout vs Deep Ensembles — 1-D sandbox

**Strand 4 — Deep ensembles.**

A 1-D regression on noisy `y = x * sin(x)` with a deliberate **gap** in the
training range (`x ∈ [-1, 1]` is excluded). The sandbox checks whether each
uncertainty method reports high variance *inside the gap* — the property a
UCB-style exploration term needs to behave sensibly.

Models and training helpers live in `src/uncertainty/models.py`.

## Imports

In [ ]:
import sys, pathlib

_repo_root = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "requirements.txt").exists()
)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.uncertainty.models import (
    generate_data,
    train_dropout_model, predict_mc_dropout,
    train_ensemble, predict_ensemble,
)

## Parameters

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Seed: {SEED}")

## Data

In [ ]:
x_raw, y_raw, x_train, y_train = generate_data()

# Test range covers the gap (-1, 1) and extends slightly past training bounds.
x_test = np.linspace(-5, 5, 200)
x_test_tensor = torch.FloatTensor(x_test).unsqueeze(1)

## MC Dropout

In [ ]:
dropout_model = train_dropout_model(x_train, y_train)
mc_mean, mc_std = predict_mc_dropout(dropout_model, x_test_tensor)

## Deep Ensemble

In [ ]:
ensemble_models = train_ensemble(x_train, y_train)
ens_mean, ens_std = predict_ensemble(ensemble_models, x_test_tensor)

## Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot MC Dropout
ax1.scatter(x_raw, y_raw, color='black', s=10, label='Data')
ax1.plot(x_test, mc_mean, color='blue', label='Mean Prediction')
# UCB-style confidence interval: Mean + k * sigma
ax1.fill_between(x_test, mc_mean - 2*mc_std, mc_mean + 2*mc_std, color='blue', alpha=0.2, label='Uncertainty (2$\sigma$)')
ax1.set_title('MC Dropout Uncertainty')
ax1.set_ylim(-6, 6)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot Ensemble
ax2.scatter(x_raw, y_raw, color='black', s=10, label='Data')
ax2.plot(x_test, ens_mean, color='green', label='Mean Prediction')
ax2.fill_between(x_test, ens_mean - 2*ens_std, ens_mean + 2*ens_std, color='green', alpha=0.2, label='Uncertainty (2$\sigma$)')
ax2.set_title('Deep Ensemble Uncertainty')
ax2.set_ylim(-6, 6)
ax2.legend()
ax2.grid(True, alpha=0.3)

print("\nVisualization generated. Notice how uncertainty expands in the gap (-1 to 1) and outside training bounds.")
plt.tight_layout()
plt.show()